Here is how it works:
- Full Clone and extract all config files: .yml, .yaml and related .json, .sh
- Shallow Clone from all branches: afect number commits & contributors build script or config will be only from the latest snapshop

- it sorts and index the url also save the list of random sample url indeces
- after each puase or interuption the "Cloned Repo" should be emptyied
- by a new new rerun it continues the review from the last reviewed url which is log is stored in .evn by START_NUMBER
- the sample repos are stored in "Cloned_Sample"
- this will save the sample repos as well as metrics, configs, builds and test lines
- Full Clone helps to extract full contributors and commit history
- saving metadata happend immediately so it will get lost by pause/start
Extre feature in v2.0:
- it does compare the downloaded yml files with the list from the previous step


In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import stat
import re

# === CONFIGURATION ===
MAX_PROJECTS = 3581
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)

# Load all available GitHub tokens
TOKENS = [os.getenv(f'GITHUB_TOKEN_{i}') for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]

if not TOKENS:
    raise ValueError("❌ No GitHub tokens found in All_tokens.env")

token_index = 0  # For rotation

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\URL_List.csv")
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "Project_Metadata.csv"
config_location_csv = base_dir / "Config_Location.csv"
git_metadata_dir = base_dir / "Git_Metadata"

# === ENSURE ALL FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir, git_metadata_dir]:
    path.mkdir(parents=True, exist_ok=True)

# CI_Services Lock down list
ci_patterns = {
    r'\.travis\.yml$': 'Travis CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'CircleCI',
    r'\.circleci/config\.yml$': 'CircleCI',
    r'azure-pipelines\.yml$': 'Azure Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === LOAD EXISTING CONFIG LOCATIONS IF RESUMING ===
if config_location_csv.exists():
    config_locations_df = pd.read_csv(config_location_csv)
else:
    config_locations_df = pd.DataFrame(columns=["repo_name", "config_file_path", "file_type"])

# === COMMIT METADATA EXTRACTION FUNCTION ===
def extract_commit_metadata(repo_path, output_path):
    try:
        cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
        result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
        commit_hashes = result_hashes.stdout.strip().split("\n")

        rows = []
        for commit in commit_hashes:
            cmd_metadata = ["git", "-C", str(repo_path), "show", "--quiet",
                            f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit]
            result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True)
            if result_metadata.stdout:
                parts = result_metadata.stdout.strip().split("|", maxsplit=4)
                if len(parts) < 5:
                    continue
            else:
                continue

            cmd_files = ["git", "-C", str(repo_path), "show", "--name-only", "--pretty=format:", commit]
            result_files = subprocess.run(cmd_files, capture_output=True, text=True, check=True)
            changed_files = result_files.stdout.strip().split("\n")
            changed_files = [f.strip() for f in changed_files if f.strip()]

            count_androidTest = sum("androidTest" in f for f in changed_files)
            count_github_workflows = sum(".github/workflows" in f for f in changed_files)
            count_gradle = sum("build.gradle" in f for f in changed_files)

            rows.append({
                "commit_hash": parts[0],
                "author_name": parts[1],
                "author_email": parts[2],
                "commit_date": parts[3],
                "commit_message": parts[4],
                "touches_androidTest": count_androidTest > 0,
                "count_androidTest": count_androidTest,
                "touches_github_workflows": count_github_workflows > 0,
                "count_github_workflows": count_github_workflows,
                "touches_gradle": count_gradle > 0,
                "count_gradle": count_gradle
            })

        if rows:
            df = pd.DataFrame(rows)
            output_path.mkdir(parents=True, exist_ok=True)
            df.to_csv(output_path / "contributors_commits.csv", index=False)
    except subprocess.CalledProcessError:
        pass

# === FUNCTION TO HANDLE READ-ONLY FILES ===
def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

# === FUNCTION TO GET COUNT FROM GITHUB API ===
def get_count(api_url, headers):
    try:
        r = requests.get(api_url, headers=headers, params={"per_page": 100})
        if r.status_code != 200:
            return 0
        return len(r.json())
    except:
        return 0

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.iloc[i]['github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', url, str(repo_path)],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception as e:
        continue

    # === Detect and checkout default branch ===
    try:
        base_api = f"https://api.github.com/repos/{username}/{project}"
        headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
        token_index += 1
        r_branch = requests.get(base_api, headers=headers, timeout=15)
        if r_branch.status_code == 200:
            default_branch = r_branch.json().get('default_branch', 'main')
            subprocess.run(["git", "-C", str(repo_path), "checkout", default_branch],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except:
        pass

    try:
        result = subprocess.run(['git', '-C', str(repo_path), 'rev-list', '--count', 'HEAD'],
                                capture_output=True, text=True, check=True)
        local_commit_count = int(result.stdout.strip())
    except subprocess.CalledProcessError:
        local_commit_count = 0

    if local_commit_count > 0:
        extract_commit_metadata(repo_path, git_metadata_dir / repo_name)

    # === Scan and copy config/build files ===
    ci_keywords = ['ci', 'build', 'test', 'workflow', 'pipeline', 'instrumentation']
    config_files_found = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            try:
                should_copy = False
                file_type = file_lower.split('.')[-1]

                ci_platform = "Other"
                for pattern, platform in ci_patterns.items():
                    if re.search(pattern, rel_path, re.IGNORECASE):
                        ci_platform = platform
                        break

                if file_lower.endswith(('.yml', '.yaml')):
                    matched_ci_type = None
                    for pattern, platform in ci_patterns.items():
                        if re.search(pattern, rel_path, re.IGNORECASE):
                            matched_ci_type = platform
                            break
                    if matched_ci_type:
                        should_copy = True
                        ci_platform = matched_ci_type

                elif file_lower.endswith('build.gradle'):
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        if any(keyword in f.read().lower() for keyword in ['test', 'instrumentation']):
                            should_copy = True

                elif file_lower.endswith(('.json', '.sh')):
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        if any(keyword in f.read().lower() for keyword in ci_keywords):
                            should_copy = True

                if should_copy:
                    rel_parts = rel_path.replace("/", ".").replace("\\", ".")
                    flat_filename = f"{username}.{project}.{ci_platform}.{file}"
                    destination_path = build_info_dir / flat_filename if file_lower.endswith('build.gradle') else yml_output_dir / flat_filename
                    shutil.copy2(file_path, destination_path)
                    config_files_found.append({
                        "repo_name": repo_name,
                        "config_file_path": flat_filename,
                        "file_type": file_type
                    })
            except:
                pass

    if config_files_found:
        config_locations_df = pd.concat([config_locations_df, pd.DataFrame(config_files_found)], ignore_index=True)

    # === YML COMPARISON LOGIC INJECTION START ===
    try:
        yml_reference_df = pd.read_csv(r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_detected_yml_files.csv")
        downloaded_yml_df = pd.DataFrame(config_files_found)
        downloaded_yml_df['match_in_reference'] = False
        downloaded_yml_df['match_reason'] = ''
        repo_url = url.strip().rstrip('/')
        for idx, row in downloaded_yml_df.iterrows():
            matched = yml_reference_df[
                (yml_reference_df['html_url'].str.strip().str.rstrip('/') == repo_url) &
                (yml_reference_df['file_path'].str.strip().str.replace('\\', '/').str.lower() ==
                 row['config_file_path'].split('.', 2)[-1].lower())
            ]
            if not matched.empty:
                downloaded_yml_df.at[idx, 'match_in_reference'] = True
            else:
                downloaded_yml_df.at[idx, 'match_reason'] = 'New file not in step4_detected_yml_files.csv'

        missing_ymls = yml_reference_df[yml_reference_df['html_url'].str.strip().str.rstrip('/') == repo_url]
        found_paths = [f['config_file_path'].split('.', 2)[-1].lower() for f in config_files_found]
        missing_ymls = missing_ymls[~missing_ymls['file_path'].str.lower().isin(found_paths)]

        for _, row in missing_ymls.iterrows():
            downloaded_yml_df = pd.concat([downloaded_yml_df, pd.DataFrame([{
                'repo_name': repo_name,
                'config_file_path': row['file_path'],
                'file_type': 'yml',
                'match_in_reference': False,
                'match_reason': 'Expected but not found in cloned repo'
            }])], ignore_index=True)

        match_csv = base_dir / 'YML_Download_Match.csv'
        downloaded_csv = base_dir / 'YML_Files_Cloned.csv'

        downloaded_yml_df.to_csv(match_csv, mode='a' if match_csv.exists() else 'w', index=False, header=not match_csv.exists())
        pd.DataFrame(config_files_found).to_csv(downloaded_csv, mode='a' if downloaded_csv.exists() else 'w', index=False, header=not downloaded_csv.exists())
    except Exception as e:
        print(f"⚠️ YML check failed: {e}")
    # === YML COMPARISON LOGIC INJECTION END ===

    # Cleanup or keep sample
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
        else:
            shutil.rmtree(repo_path, onerror=force_remove_readonly)
    except:
        pass

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))
    config_locations_df.drop_duplicates().to_csv(config_location_csv, index=False)

print(f"\n✅ Process complete. Sampled: {len(sample_indices_to_keep)} | Total Processed: {len(df) - (START_NUMBER - 1)}")
print("\n✅ All selected repositories have been processed.")


🔁 Loaded SAMPLE_LIST from .env with 150 indices.

🔍 [10/3581] Processing 0009.RHVoice.RHVoice...
✅ Clone complete


Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 56: character maps to <undefined>


📌 Checked out default branch: master
⚠️ Skipped malformed commit in 0009.RHVoice.RHVoice (missing metadata)
⚠️ No commit data for 0009.RHVoice.RHVoice
📜 Metadata saved
👥 Saved contributors to: RHVoice.RHVoice.contributors.txt
🕵️ Deleted cloned repo: 0009.RHVoice.RHVoice

🔍 [11/3581] Processing 0010.NXT.LEGO-MINDSTORMS-MINDdroid...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata for 0010.NXT.LEGO-MINDSTORMS-MINDdroid
📜 Metadata saved
👥 Saved contributors to: NXT.LEGO-MINDSTORMS-MINDdroid.contributors.txt
🕵️ Deleted cloned repo: 0010.NXT.LEGO-MINDSTORMS-MINDdroid

🔍 [12/3581] Processing 0011.opendocument-app.OpenDocument.droid...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 0011.opendocument-app.OpenDocument.droid
📜 Metadata saved
👥 Saved contributors to: opendocument-app.OpenDocument.droid.contributors.txt
🕵️ Deleted cloned repo: 0011.opendocument-app.OpenDocument.droid

🔍 [13/3581] Processing 0012.drawpile.Drawpile...
❌ C

Exception in thread Thread-965 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 100: character maps to <undefined>


📌 Checked out default branch: master
⚠️ Skipped malformed commit in 0133.halzhang.StartupNews (missing metadata)
⚠️ No commit data for 0133.halzhang.StartupNews
📜 Metadata saved
👥 Saved contributors to: halzhang.StartupNews.contributors.txt
🕵️ Deleted cloned repo: 0133.halzhang.StartupNews

🔍 [135/3581] Processing 0134.pR0Ps.LocationShare...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata for 0134.pR0Ps.LocationShare
📜 Metadata saved
👥 Saved contributors to: pR0Ps.LocationShare.contributors.txt
🕵️ Deleted cloned repo: 0134.pR0Ps.LocationShare

🔍 [136/3581] Processing 0135.enviroCar.enviroCar-app...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata for 0135.enviroCar.enviroCar-app
📜 Metadata saved
👥 Saved contributors to: enviroCar.enviroCar-app.contributors.txt
🕵️ Deleted cloned repo: 0135.enviroCar.enviroCar-app

🔍 [137/3581] Processing 0136.guardianproject.CameraV...
✅ Clone complete
📌 Checked out default branch: master
✅ Save

...

📌 Checked out default branch: main
⚠️ Skipped malformed commit in 3486.koofr.vault (missing metadata)
⚠️ No commit data for 3486.koofr.vault
📜 Metadata saved
👥 Saved contributors to: koofr.vault.contributors.txt
🕵️ Deleted cloned repo: 3486.koofr.vault

🔍 [3488/3581] Processing 3487.cardano-foundation.veridian-wallet...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 3487.cardano-foundation.veridian-wallet
📜 Metadata saved
👥 Saved contributors to: cardano-foundation.veridian-wallet.contributors.txt
🕵️ Deleted cloned repo: 3487.cardano-foundation.veridian-wallet

🔍 [3489/3581] Processing 3488.brumeproject.wallet...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 3488.brumeproject.wallet
📜 Metadata saved
👥 Saved contributors to: brumeproject.wallet.contributors.txt
🕵️ Deleted cloned repo: 3488.brumeproject.wallet

🔍 [3490/3581] Processing 3489.fruitriin.missRirica-client...
✅ Clone complete
📌 Checked out default branch: v2-m

Exception in thread Thread-27061 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 107: character maps to <undefined>


📌 Checked out default branch: main
⚠️ Skipped malformed commit in 3553.KiWi233333.JiwuChat (missing metadata)
⚠️ No commit data for 3553.KiWi233333.JiwuChat
📜 Metadata saved
👥 Saved contributors to: KiWi233333.JiwuChat.contributors.txt
🕵️ Deleted cloned repo: 3553.KiWi233333.JiwuChat

🔍 [3555/3581] Processing 3554.flomesh-io.ztm...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 3554.flomesh-io.ztm
📜 Metadata saved
👥 Saved contributors to: flomesh-io.ztm.contributors.txt
🕵️ Deleted cloned repo: 3554.flomesh-io.ztm

🔍 [3556/3581] Processing 3555.mym0404.react-native-naver-map...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 3555.mym0404.react-native-naver-map
📜 Metadata saved
👥 Saved contributors to: mym0404.react-native-naver-map.contributors.txt
🕵️ Deleted cloned repo: 3555.mym0404.react-native-naver-map

🔍 [3557/3581] Processing 3556.bitcoinppl.cove...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commi

Exception in thread Thread-27163 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 130: character maps to <undefined>


📌 Checked out default branch: main
⚠️ Skipped malformed commit in 3566.Axixi2233.chiaki-android (missing metadata)
⚠️ No commit data for 3566.Axixi2233.chiaki-android
📜 Metadata saved
👥 Saved contributors to: Axixi2233.chiaki-android.contributors.txt
🕵️ Deleted cloned repo: 3566.Axixi2233.chiaki-android

🔍 [3568/3581] Processing 3567.a-ghorbani.pocketpal-ai...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 3567.a-ghorbani.pocketpal-ai
📜 Metadata saved
👥 Saved contributors to: a-ghorbani.pocketpal-ai.contributors.txt
🕵️ Deleted cloned repo: 3567.a-ghorbani.pocketpal-ai

🔍 [3569/3581] Processing 3568.Jellify-Music.App...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata for 3568.Jellify-Music.App
📜 Metadata saved
👥 Saved contributors to: Jellify-Music.App.contributors.txt
🕵️ Deleted cloned repo: 3568.Jellify-Music.App

🔍 [3570/3581] Processing 3569.google-ai-edge.LiteRT...
✅ Clone complete
📌 Checked out default branch: main
✅ Sav